# What changes on a twenty-first birthday?

**A guided investigation · Chapter 4 · 30–45 minutes · No Python experience needed**

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cmg777/intro2causal/blob/main/notebooks_colab/explore-mlda_clean.ipynb)

> **The question**  
> Do mortality patterns change at the age when alcohol becomes legally available?

A birthday is an ordinary passage of time, but turning 21 in the United States also changes legal access to alcohol. Carpenter and Dobkin used this threshold to investigate the relationship between alcohol access and mortality. The teaching file records mortality rates for narrow age cells around the cutoff rather than individual people. Each point on a chart is therefore an aggregate, measured in deaths per 100,000 person-years. The file contains both actual age and age centered on 21; confusing the two would put the policy threshold in the wrong place. We will plot actual age, compare causes of death, and zoom in near the birthday. The central reasoning challenge is to distinguish a gradual age pattern from an abrupt change at the policy boundary, while asking what else could change at precisely the same age.

### By the end, you will be able to…

- Use an age threshold to ask when nearby observations might offer a useful comparison.
- Read and modify three interactive charts using short Python examples.
- Explain one reason a descriptive pattern alone cannot settle this case’s causal question.

### Before you see the evidence

Would you expect mortality to change smoothly with age, or jump at a particular birthday?

Write two sentences now. You will return to them at the end.

**Your route:** meet the data → inspect three views → modify and try → ask the causal question.

## Your first few minutes in Colab · 3 minutes

You do not need prior Python experience. A **cell** is one block of text or code. Read the text, then press the **▶** beside the next code cell (or Shift+Enter). Its output appears underneath. Run cells in order: later cells use tables created earlier.

Choose **File → Save a copy in Drive** if you want to keep your changes. If a name is “not defined,” run the earlier setup cells again. **Runtime → Restart session and run all** gives you a fresh start. An error does not mean you have damaged the data: this notebook only reads it.

Keep a short learning log in a text cell. Before each chart, make a prediction. Afterward, write one observation with a number and one question that the chart cannot answer.

**Chart controls:** hover for values, click legend labels to hide/show a group, drag to zoom, and double-click to reset. A chart’s menu also offers a reset/autoscale control.

The setup installs Plotly Express if needed. `import` makes a library available; `pd` and `px` are short names for pandas (tables) and Plotly Express (charts).

In [ ]:
%pip install -q "plotly>=6,<7"

In [ ]:
import pandas as pd
import plotly.express as px
from IPython.display import display

# One consistent chart style; no plotting experience needed.
px.defaults.template = "plotly_white"
px.defaults.color_continuous_scale = "Blues"
px.defaults.color_discrete_sequence = ["#2563eb", "#7c3aed", "#059669", "#64748b"]

## Meet the data · 5 minutes

**One row:** Monthly age cell.  
**Stored size:** 50 rows × 13 columns.

| Column | What it records |
|---|---|
| `agecell` | Actual age in years for an aggregate mortality cell. |
| `all` | All-cause mortality rate, deaths per 100,000 person-years. |
| `mva` | Motor-vehicle mortality rate, deaths per 100,000 person-years. |
| `suicide` | Suicide mortality rate, deaths per 100,000 person-years. |
| `over21` | Indicator for the age-21 cutoff side: 1 at or above the threshold. |

Use agecell for actual age; age is centered on 21. Rates describe age cells, not individual death probabilities. A visible jump alone does not establish the assumptions of a causal design.

`read_csv` downloads a table. We call it `df`. `head(5)` shows just five rows, not a separate sample used for the analysis. Square brackets select columns by their names.

In [ ]:
DATA_URL = "https://raw.githubusercontent.com/cmg777/intro2causal/main/data/ch4/mlda_clean.csv"
df = pd.read_csv(DATA_URL)
print(f"Loaded {len(df):,} rows and {len(df.columns)} columns")
display(df[['agecell', 'all', 'mva', 'suicide', 'over21']].head(5))

### Pause: what do the rows mean?

Choose one displayed row. Translate it into a sentence using the variable guide. Which values are categories or identifiers? Which are quantities? A numeric storage type does not necessarily make a code a meaningful measurement.

The next table checks types, missing values, and the number of distinct values for the key columns. `isna()` marks missing cells; `sum()` counts them. Missing is not zero. Ask which comparison would lose a row if one of its required values were missing.

In [ ]:
key_columns = ['agecell', 'all', 'mva', 'suicide', 'over21']
checks = pd.DataFrame({"storage_type": df[key_columns].dtypes.astype(str),
                       "missing_rows": df[key_columns].isna().sum(),
                       "distinct_values": df[key_columns].nunique()})
display(checks)
print("The charts below omit missing values required for each plotted quantity.")

### A small summary before plotting

`describe()` summarizes numeric columns: `count` counts available values, `mean` is the average, and `50%` is the median. The median splits ordered values into two halves. Averages of identifiers or arbitrary category codes are not useful descriptions, even if Python calculates them.

For category-like key variables, the second table counts values, including missing entries. Check these labels before interpreting a chart.

In [ ]:
numeric_keys = df[key_columns].select_dtypes(include="number").columns
if len(numeric_keys):
    display(df[numeric_keys].describe().round(2))
for column in key_columns:
    if df[column].nunique() <= 8:
        display(df[column].value_counts(dropna=False).rename("Rows").to_frame())

## Read the chart code · 2 minutes

`px.bar`, `px.scatter`, `px.line`, and `px.box` choose a chart type. `x` and `y` name columns for the axes; `color` distinguishes groups or values. `labels` translates column names for readers. `fig.show()` displays the result.

A table transformation and a plot answer different questions: first decide what each plotted row represents, then choose how to display it. Before running each chart, predict what you expect to see.

In [ ]:
# Display labels: run once; you do not need to memorize this mapping.
labels = {'ability_group': 'Ability group',
 'age': 'Age (years)',
 'agecell': 'Age (years)',
 'all': 'Deaths per 100,000 person-years',
 'census': 'Census year',
 'group': 'State group',
 'mva': 'Motor-vehicle deaths per 100,000 person-years',
 'offer': 'Scholarship assignment',
 'plan': 'Assigned insurance plan',
 'state_name': 'Simulated state'}

## Visual investigation 1 · Locate the policy boundary

**Predict:** What happens to all-cause mortality near age 21?

A scatter plot keeps the age-cell observations visible. We draw a vertical guide at actual age 21; the guide is a policy threshold, not a fitted causal effect.

In [ ]:
fig = px.scatter(df, x="agecell", y="all", labels=labels, title="Mortality around the twenty-first birthday")
fig.add_vline(x=21, line_dash="dash", line_color="#7c3aed")
fig.show()

**Explore the chart:** Hover just below and above 21. Describe the local pattern rather than comparing the youngest and oldest cells.

**Write before reading on:** Describe one pattern with its unit and comparison group. Then give one possible explanation.

<details>
<summary>Interpretation checkpoint</summary>

A local change suggests a question for regression discontinuity. Whether it identifies an alcohol-access effect depends on other influences being sufficiently smooth at the boundary.

</details>

## Visual investigation 2 · Ask which outcomes move

**Predict:** Do different causes of death have the same age pattern?

melt turns separate outcome columns into one rate column and one cause label. Separate panels preserve each cause’s shape without confusing the measures.

In [ ]:
causes = df.melt(id_vars="agecell", value_vars=["mva", "suicide", "internal"], var_name="cause", value_name="rate")
fig = px.scatter(causes, x="agecell", y="rate", facet_col="cause", facet_col_wrap=2, labels={"agecell": "Age (years)", "rate": "Deaths per 100,000 person-years"}, title="Compare causes of death")
fig.add_vline(x=21, line_dash="dash")
fig.show()

**Explore the chart:** Compare motor-vehicle (mva), suicide, and internal causes. Do all panels suggest an equally abrupt change?

**Write before reading on:** Describe one pattern with its unit and comparison group. Then give one possible explanation.

<details>
<summary>Interpretation checkpoint</summary>

A design should make sense for specific outcomes. Different cause patterns can help investigate mechanisms, but do not by themselves eliminate alternative explanations.

</details>

## Visual investigation 3 · Change the window

**Predict:** Does the pattern survive a closer look at the cutoff?

between selects observations in a stated interval. A smaller window gives a more local comparison but fewer data points.

In [ ]:
near = df[df["agecell"].between(20.5, 21.5)]
fig = px.scatter(near, x="agecell", y="all", labels=labels, title="A one-year window around the threshold")
fig.add_vline(x=21, line_dash="dash")
fig.show()

**Explore the chart:** Count how many points remain on each side. Compare this view with the full age range.

**Write before reading on:** Describe one pattern with its unit and comparison group. Then give one possible explanation.

<details>
<summary>Interpretation checkpoint</summary>

Choosing a window involves a tradeoff between local comparability and information. Zooming is a diagnostic, not a substitute for estimating and checking a research design.

</details>

## Make it yours · 8 minutes

### 1. Modify an example

Change the window to 20.75–21.25 and notice how much information is lost.

Copy that chart cell before editing, so you can compare the original and modified outputs. Say what changed in your question as well as in your code.

### 2. Try a small investigation

Plot motor-vehicle mortality within ages 20–22 and mark age 21.

Write a prediction first. The optional starter cell below is safe to leave unchanged; later cells do not depend on your answer.

In [ ]:
# Your investigation goes here.
# Describe your expected pattern before writing code.

<details>
<summary>Hint 1 · choose the right evidence</summary>

Filter using agecell, not the centered age column.

</details>

<details>
<summary>Hint 2 · a Python starting point</summary>

Use px.scatter(..., y="mva") and add_vline(x=21).

</details>

<details>
<summary>Optional worked solution · open after trying</summary>

<pre><code>near = df[df[&quot;agecell&quot;].between(20, 22)]
fig = px.scatter(near, x=&quot;agecell&quot;, y=&quot;mva&quot;, labels=labels)
fig.add_vline(x=21, line_dash=&quot;dash&quot;)
fig.show()</code></pre>

Describe a visible pattern cautiously. You cannot infer an individual person’s risk or rule out every coincident change from these cells alone.

Copy the code into the starter cell to run it.

</details>

## What can you now say? · 4 minutes

Return to your initial prediction. Write a four-sentence evidence note:

1. **I expected…** State your prediction.
2. **I observed…** Use a chart, a comparison, and the correct units.
3. **This does not establish…** Name a causal claim the evidence cannot settle.
4. **To learn more, I would need…** Propose an additional measurement or a more credible comparison.

### Carry this question into the chapter

**What must be true about other influences near age 21 for the cutoff to help isolate legal alcohol access?**

[Continue with Chapter 4: Regression Discontinuity](https://cmg777.github.io/intro2causal/book/_book/notebooks_quarto/04-regression-discontinuity.html#the-mlda-question) · [Return to the case collection](https://cmg777.github.io/intro2causal/data-dictionary.html#ch4-mlda-clean-csv)

### Source and preparation notes

[Carpenter and Dobkin (2009), The Effect of Alcohol Consumption on Mortality](https://www.aeaweb.org/articles?id=10.1257/app.1.1.164)

The preparation script creates centered age, the age-21 indicator, cause-specific rates, external-other mortality by subtraction, and polynomial/interacted terms. Rates are source age-cell aggregates; no individual records are supplied. Mortality rates are deaths per 100,000 person-years.

These exploratory charts are based on the committed CSV. They are not a complete replication of the original study. Keep the source study, this prepared extract, and your descriptive results distinct.